# 06 - DPO 对齐对比: 手写 DPOTrainer vs TRL DPOTrainer

对比 from-scratch DPOTrainer (手写 DPO loss) 与 TRL DPOTrainer 的实现差异。

| 维度 | from-scratch | HuggingFace (TRL) |
|------|-------------|-------------------|
| Ref Model | `copy.deepcopy(model)` 手动冻结 | DPOTrainer 自动管理 |
| 数据格式 | 手动 tokenize 4 组 tensor | 字符串格式, 自动 tokenize |
| Log Probs | `log_softmax + gather + mask` | 内置计算 |
| DPO Loss | `-logsigmoid(beta * margin).mean()` | 内置多种 loss 变体 |
| 代码量 | ~400 行 (DPOTrainer class) | ~40 行 (run_dpo function) |

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import tempfile
from pathlib import Path

from data import ClearMindTokenizer, load_dpo_dataset

## 1. DPO 数据格式对比

In [ ]:
# 准备 tokenizer 和数据
tokenizer_path = '../outputs/tokenizer'
if Path(tokenizer_path).exists():
    tokenizer = ClearMindTokenizer.load(tokenizer_path)
else:
    tmpdir = tempfile.mkdtemp()
    corpus = Path(tmpdir) / 'corpus.txt'
    corpus.write_text('\n'.join(['深度学习。', 'AI技术。', 'Transformer.'] * 20))
    tokenizer = ClearMindTokenizer.train(str(corpus), vocab_size=500)

tmpdir = tempfile.mkdtemp()
dpo_path = Path(tmpdir) / 'dpo.jsonl'
with open(dpo_path, 'w') as f:
    for item in [
        {'prompt': '什么是AI？', 'chosen': 'AI是计算机科学的分支。', 'rejected': 'AI就是电脑。'},
        {'prompt': '什么是NLP？', 'chosen': 'NLP是处理人类语言的技术。', 'rejected': 'NLP是编程。'},
    ]:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# === HuggingFace 方式: 字符串格式 ===
dpo_ds = load_dpo_dataset(str(dpo_path), tokenizer, max_length=128)
sample = dpo_ds['train'][0]

print('=== HuggingFace DPO 数据格式 ===')
print(f'类型: 纯字符串 (DPOTrainer 内部自动 tokenize)')
print(f'  prompt:   {repr(sample["prompt"])}')
print(f'  chosen:   {repr(sample["chosen"])}')
print(f'  rejected: {repr(sample["rejected"])}')

print('\n=== from-scratch DPO 数据格式 ===')
print('类型: 预 tokenize 的 tensor')
print('  chosen_input_ids:   tensor([1, 29, 281, ...])  # 已 tokenize')
print('  chosen_labels:      tensor([-100, -100, ..., 45, 124, ...])  # 含 loss mask')
print('  rejected_input_ids: tensor([1, 29, 281, ...])  # 已 tokenize')
print('  rejected_labels:    tensor([-100, -100, ..., 73, 89, ...])  # 含 loss mask')

## 2. Ref Model 管理对比

In [ ]:
print('=== from-scratch: 手动管理 ref_model ===')
print('''
import copy

# 训练开始前创建 ref_model
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# 训练时手动计算 ref_model 的 log probs
with torch.no_grad():
    ref_logits, _, _ = ref_model(input_ids)
    ref_logps = compute_log_probs(ref_logits, labels, mask)
''')

print('=== HuggingFace (TRL): DPOTrainer 自动管理 ===')
print('''
import copy

# 显式传入 ref_model (自定义模型需要)
ref_model = copy.deepcopy(model)
ref_model.eval()

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,  # DPOTrainer 自动管理冻结和推理
    ...
)
# DPOTrainer 内部自动:
#   - 冻结 ref_model 参数
#   - 前向传播计算 ref log probs
#   - 计算 policy vs ref 的 margin
''')

## 3. DPO Loss 计算对比

In [ ]:
print('=== from-scratch: 手写 DPO Loss ===')
print('''
def compute_log_probs(logits, labels, mask):
    """计算每个 token 的 log probability"""
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return (token_log_probs * mask).sum(-1) / mask.sum(-1)

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """DPO loss: -log_sigmoid(beta * margin)"""
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)
    margin = chosen_rewards - rejected_rewards
    loss = -F.logsigmoid(margin).mean()
    return loss
''')

print('=== HuggingFace (TRL): 内置 DPO Loss ===')
print('''
# TRL DPOTrainer._compute_loss() 内部完成:
# 1. 前向传播 model 和 ref_model
# 2. 计算 per-token log probs (selective_log_softmax)
# 3. 应用 completion_mask
# 4. 计算 DPO loss (支持 sigmoid/hinge/ipo 等变体)
# 5. 记录 metrics (chosen_rewards, rejected_rewards, margin)
#
# 用户只需:
trainer = DPOTrainer(model=model, ref_model=ref_model, args=DPOConfig(beta=0.1))
trainer.train()  # 一行完成
''')

## 4. 完整管线对比

In [ ]:
print('=== HuggingFace 完整 Pretrain → SFT → DPO 管线 ===')
print('''
# Step 1: 预训练
python scripts/train.py --stage pretrain --config configs/tiny.yaml

# Step 2: SFT (从预训练模型加载)
python scripts/train.py --stage sft --config configs/tiny.yaml \\
    --resume outputs/pretrain

# Step 3: DPO (从 SFT 模型加载)
python scripts/train.py --stage dpo --config configs/tiny.yaml \\
    --resume outputs/sft

# 每个阶段的模型都用 from_pretrained / save_pretrained 传递
# 格式统一: config.json + model.safetensors
''')

## 总结

| 功能 | from-scratch | HuggingFace (TRL) |
|------|-------------|-------------------|
| 数据格式 | 4 组 tensor (手动 tokenize) | 3 个字符串 (自动 tokenize) |
| Ref Model | `copy.deepcopy` + 手动冻结 | `ref_model` 参数，自动管理 |
| Log Probs | `log_softmax + gather + mask` | `selective_log_softmax` 内置 |
| DPO Loss | 手写 `-logsigmoid(beta*margin)` | 内置 sigmoid/hinge/ipo 变体 |
| 训练循环 | 手写 epoch + micro-batch | `DPOTrainer.train()` |
| Metrics | 手动计算 accuracy/margin | 自动记录 rewards/margin/accuracy |
| 代码量 | ~400 行 | ~40 行 |

**核心收获:** DPO 的数学原理完全相同，
TRL DPOTrainer 将复杂的 ref model 管理、log prob 计算、loss 计算全部封装，
用户只需提供 `{prompt, chosen, rejected}` 字符串格式数据即可训练。